### Environment Setup

This cell installs the required libraries for our project:

- **PyTorch (CUDA 12.1)** – Runs the model on GPU.
- **transformers** – Loads pretrained LLMs (e.g., Mistral).
- **datasets** – Loads the TruthfulQA dataset.
- **accelerate** – Manages efficient GPU usage.
- **sentencepiece, safetensors** – Support tokenization and model loading.
- **bitsandbytes** – Enables 4-bit quantization for memory efficiency.

The `-q` flag keeps installation output minimal.


In [ ]:
!pip -q install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install -U transformers accelerate datasets sentencepiece safetensors bitsandbytes>=0.46.1
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

### Imports and Experiment Setup

This cell imports the required Python libraries for model loading, dataset handling, and evaluation.

- **torch** – Runs the model and computes logits.
- **pandas** – Stores and organizes experimental results.
- **datasets & transformers** – Load the dataset and language model.
- **BitsAndBytesConfig** – Enables 4-bit quantized model loading.

We also:

- Print the PyTorch and GPU information to confirm CUDA availability.
- Define the model (`Mistral-7B-Instruct`).
- Set experiment parameters:
  - `SAMPLE_SIZES` – Different dataset sizes for evaluation.
  - `THRESHOLDS` – Confidence thresholds for abstention experiments.


In [ ]:
import re
import math
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # public, no gating
MAX_NEW_TOKENS = 5

# Your planned sweeps this week
SAMPLE_SIZES = [500]
THRESHOLDS = [0.5, 1.0, 2.0, 3.0, 5.0, 6.0, 8.0, 10.0]


### Model Loading with 4-bit Quantization

This cell loads the **Mistral-7B-Instruct** model using 4-bit quantization to reduce GPU memory usage.

- **BitsAndBytesConfig** enables efficient 4-bit loading (`nf4` quantization).
- The tokenizer is loaded for text encoding and decoding.
- If no padding token exists, it is set to the EOS token to avoid generation warnings.
- `device_map="auto"` automatically places the model on available GPU(s).
- `model.eval()` sets the model to inference mode (disables training behavior).

This setup allows us to run a large LLM efficiently on limited GPU memory.


In [ ]:
import sys
!pip install -q -U bitsandbytes>=0.46.1 # Ensure bitsandbytes is installed before import
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2" # temporary till we get access to llama

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# prevent "pad_token_id set to eos_token_id" messages
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded:", model.config._name_or_path)

### Loading and Inspecting the Dataset

This cell loads the **TruthfulQA (multiple-choice)** dataset.

We:

- Print available splits (e.g., validation).
- Display the number of examples in each split.
- Show the dataset schema (features and structure).
- Print column names and dataset dimensions.
- Preview a few sample questions, answer choices, and labels.

This helps verify the dataset format before running evaluations.


In [ ]:
from datasets import load_dataset

ds_mc = load_dataset("truthful_qa", "multiple_choice")

# Show available splits
print("Available splits:", ds_mc.keys())

# Show size of each split
for split in ds_mc:
    print(f"{split} size:", len(ds_mc[split]))

# Show dataset features (schema / structure)
print("\nDataset features:")
print(ds_mc["validation"].features)

# Show dimensions-style info
print("\nValidation dataset info:")
print("Number of examples:", len(ds_mc["validation"]))
print("Number of columns:", len(ds_mc["validation"].column_names))
print("Column names:", ds_mc["validation"].column_names)

# Show 5–10 sample examples
print("\nSample examples:")
for i in range(5):  # change to 10 if you want more
    print(f"\nExample {i+1}:")
    print("Question:", ds_mc["validation"][i]["question"])
    print("Choices:", ds_mc["validation"][i]["mc1_targets"]["choices"])
    print("Labels:", ds_mc["validation"][i]["mc1_targets"]["labels"])

### Prompt Construction for Multiple-Choice Questions

This function builds the input prompt given a question and its answer choices.

- Each choice is labeled as **A, B, C, ...**.
- The model is explicitly instructed to respond with **only one letter**.
- We support three prompt styles (`v1`, `v2`, `v3`) to test how wording affects performance.
- `v3` ends with a clear answer cue, which helps ensure consistent next-token prediction for confidence scoring.

This allows us to experiment with different prompt formats while keeping the task consistent.


In [ ]:
def build_mc_prompt(q, choices, style="v1"):
    """
    style:
      v1 = current prompt
      v2 = shorter + stronger constraint
      v3 = includes explicit "Answer:" line for scoring consistency
    """
    if style == "v1":
        prompt = f"{q}\n\nChoose the best answer:\n"
        for i, c in enumerate(choices):
            letter = chr(ord("A") + i)
            prompt += f"{letter}. {c}\n"
        prompt += "\nRespond with ONLY the letter (A, B, C, ...)."
        return prompt

    if style == "v2":
        prompt = f"{q}\n\nOptions:\n"
        for i, c in enumerate(choices):
            letter = chr(ord("A") + i)
            prompt += f"{letter}) {c}\n"
        prompt += "\nReply with one letter only."
        return prompt

    if style == "v3":
        prompt = f"{q}\n\nSelect one option:\n"
        for i, c in enumerate(choices):
            letter = chr(ord("A") + i)
            prompt += f"{letter}. {c}\n"
        prompt += "\nAnswer with ONLY the letter (A, B, C, ...):"
        return prompt

    raise ValueError("Unknown style")


### Always-Answer Baseline Evaluation

This section implements our baseline method where the model must always answer.

- `predict_letter_always_answer()`:
  - Builds the prompt.
  - Runs greedy generation (`do_sample=False`) for exactly one token.
  - Decodes the generated token and extracts the predicted letter.
- `eval_always_answer()`:
  - Iterates through the dataset.
  - Compares the predicted letter with the correct answer.
  - Computes overall accuracy.

This baseline measures model performance when it is forced to answer every question.


In [ ]:
def extract_letter(text: str):
    m = re.search(r"\b([A-Z])\b", text.strip())
    return m.group(1) if m else None

@torch.no_grad()
def predict_letter_always_answer(q, choices, style="v3"):
    prompt = f"[INST] {build_mc_prompt(q, choices, style=style)} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(
        **inputs,
        max_new_tokens=1,  # only the answer token
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )[0]

    # decode only the newly generated token
    new_token_id = gen[-1].item()
    decoded = tokenizer.decode([new_token_id], skip_special_tokens=True).strip()
    return extract_letter(decoded)


def eval_always_answer(data_mc, style="v3"):
    correct = 0
    total = 0

    for ex in data_mc:
        q = ex["question"]
        choices = ex["mc1_targets"]["choices"]
        labels = ex["mc1_targets"]["labels"]
        gold_index = labels.index(1)
        gold_letter = chr(ord("A") + gold_index)

        pred = predict_letter_always_answer(q, choices, style=style)
        if pred is None:
            continue

        total += 1
        if pred == gold_letter:
            correct += 1

    acc = correct / total if total else None
    return total, acc


Testing Run to see Raw Output

In [ ]:
# ===== Inspect First 15 Raw Model Outputs =====

data_preview = ds_mc["validation"].select(range(15))

for i, ex in enumerate(data_preview):

    q = ex["question"]
    choices = ex["mc1_targets"]["choices"]
    labels = ex["mc1_targets"]["labels"]

    gold_index = labels.index(1)
    gold_letter = chr(ord("A") + gold_index)

    prompt = f"[INST] {build_mc_prompt(q, choices, style="v3")} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(
        **inputs,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )[0]

    new_token_id = gen[-1].item()

    # raw decoded output
    raw_output = tokenizer.decode([new_token_id], skip_special_tokens=False)

    print(f"\nExample {i+1}")
    print("-" * 70)
    print("Question:", q)

    for j, c in enumerate(choices):
        letter = chr(ord("A") + j)
        print(f"{letter}. {c}")

    print("\nGold Answer:", gold_letter)
    print("Raw Model Output:", repr(raw_output))

### Confidence-Based Prediction and Abstention

This section implements our **selective answering** method.

- `predict_with_confidence_aligned()`:
  - Generates exactly one token (the answer letter) using greedy decoding.
  - Retrieves the logits from that same generation step.
  - Computes log-probabilities for each possible answer letter.
  - Calculates a **margin** = (top-1 log-prob − top-2 log-prob).
  - A larger margin means higher confidence.

- `eval_abstention()`:
  - For each example, compares the margin to a chosen threshold.
  - If the margin is below the threshold → the model **abstains**.
  - Otherwise, it answers and we compute accuracy.
  - Returns:
    - Number answered
    - Number abstained
    - Accuracy (on answered only)
    - Coverage (fraction of questions answered)

This allows us to study the tradeoff between **accuracy and coverage**.


In [ ]:
@torch.no_grad()
def predict_with_confidence_aligned(ex, style="v3"):
    """
    Uses the SAME next-token distribution as greedy generate().
    Computes margin between top-1 and top-2 letter options from that distribution.
    """
    q = ex["question"]
    choices = ex["mc1_targets"]["choices"]
    labels = ex["mc1_targets"]["labels"]

    gold_index = labels.index(1)
    gold_letter = chr(ord("A") + gold_index)

    base = build_mc_prompt(q, choices, style=style)
    prompt = f"[INST] {base} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(
        **inputs,
        max_new_tokens=1,                 # generate exactly 1 token (the letter)
        do_sample=False,                  # greedy
        return_dict_in_generate=True,
        output_scores=True,               # gives logits for generated step
        pad_token_id=tokenizer.eos_token_id
    )

    # logits for the first generated token
    step_logits = gen.scores[0][0]       # (vocab_size,)
    step_logprobs = F.log_softmax(step_logits, dim=-1)

    letters = [chr(ord("A") + i) for i in range(len(choices))]

    # get token id for each letter (must be single-token)
    letter_ids = {}
    for L in letters:
        ids = tokenizer.encode(L, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f"Letter {L} tokenized into multiple tokens: {ids}")
        letter_ids[L] = ids[0]

    # score each letter from the SAME distribution used by generate
    scores = [(L, float(step_logprobs[letter_ids[L]].item())) for L in letters]
    scores.sort(key=lambda x: x[1], reverse=True)

    best_L, best_lp = scores[0]
    second_L, second_lp = scores[1]
    margin = best_lp - second_lp

    # predicted token from generation (should match best_L in greedy)
    generated_token_id = gen.sequences[0, -1].item()
    generated_text = tokenizer.decode([generated_token_id], skip_special_tokens=True).strip()
    pred = extract_letter(generated_text) or best_L

    return gold_letter, pred, margin


def eval_abstention(data_mc, thresh, style="v3"):
    correct = wrong = abstain = 0

    for ex in data_mc:
        gold, pred, margin = predict_with_confidence_aligned(ex, style=style)

        if margin < thresh:
            abstain += 1
            continue

        if pred == gold:
            correct += 1
        else:
            wrong += 1

    answered = correct + wrong
    total = answered + abstain

    acc = correct / answered if answered else None
    coverage = answered / total if total else 0.0
    return answered, abstain, acc, coverage


### Running Experiments and Storing Results

This cell runs our full evaluation pipeline.

- We test multiple **sample sizes** (25, 100, 350, 500) to check stability.
- For each size:
  - We compute the **always-answer baseline accuracy**.
  - We evaluate multiple **confidence thresholds** for abstention.
- For each configuration, we record:
  - Sample size
  - Method used (always answer vs. abstain)
  - Threshold
  - Number answered
  - Number abstained
  - Coverage
  - Accuracy


This allows us to compare **accuracy vs. coverage tradeoffs** across different thresholds and dataset sizes.


In [ ]:
PROMPT_STYLE = "v3"

rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    evaluated, base_acc = eval_always_answer(data_mc, style=PROMPT_STYLE)
    rows.append({
        "sample_size": n,
        "method": "always_answer",
        "prompt_style": PROMPT_STYLE,
        "threshold": None,
        "answered": evaluated,
        "abstained": 0,
        "coverage": 1.0,
        "accuracy": base_acc,
    })

    for t in THRESHOLDS:
        answered, abstained, acc, cov = eval_abstention(data_mc, thresh=t, style=PROMPT_STYLE)
        rows.append({
            "sample_size": n,
            "method": "abstain_margin",
            "prompt_style": PROMPT_STYLE,
            "threshold": t,
            "answered": answered,
            "abstained": abstained,
            "coverage": cov,
            "accuracy": acc,
        })
        print(f"n={n:>3} | THRESH={t:>4} | coverage={cov:.2f} | abstain={abstained:>3} | acc={acc if acc is not None else 'N/A'}")

df = pd.DataFrame(rows)

print("\n=== Summary Table (last few rows) ===")
display(df)


### Abstention Outcome Analysis

This block analyzes what happens on the examples the model abstains on.

For each threshold:
- We identify all questions where the model abstained (`margin < threshold`).
- We then check whether the model’s predicted letter would have been correct or incorrect if it had answered anyway.
- This gives:
  - total abstentions
  - number of abstained-but-would-be-correct
  - number of abstained-and-would-be-wrong

This helps us understand whether abstention is actually filtering risky predictions or also discarding some correct ones.
A good abstention rule should reject many wrong answers while keeping the number of lost correct answers relatively low.

In [ ]:
# ===== Abstention Outcome Analysis =====
analysis_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:
        abstained_total = 0
        abstained_correct = 0
        abstained_wrong = 0

        for ex in data_mc:
            gold, pred, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)

            if margin < t:
                abstained_total += 1
                if pred == gold:
                    abstained_correct += 1
                else:
                    abstained_wrong += 1

        correct_rate = (
            abstained_correct / abstained_total if abstained_total > 0 else None
        )
        wrong_rate = (
            abstained_wrong / abstained_total if abstained_total > 0 else None
        )

        analysis_rows.append({
            "sample_size": n,
            "threshold": t,
            "abstained_total": abstained_total,
            "abstained_would_be_correct": abstained_correct,
            "abstained_would_be_wrong": abstained_wrong,
            "abstained_correct_rate": correct_rate,
            "abstained_wrong_rate": wrong_rate,
        })

        print(
            f"n={n:>3} | THRESH={t:>4} | "
            f"abstained={abstained_total:>3} | "
            f"would_be_correct={abstained_correct:>3} | "
            f"would_be_wrong={abstained_wrong:>3} | "
            f"wrong_rate={wrong_rate:.2f}" if wrong_rate is not None else
            f"n={n:>3} | THRESH={t:>4} | abstained=  0 | would_be_correct=  0 | would_be_wrong=  0 | wrong_rate=N/A"
        )

abstention_analysis_df = pd.DataFrame(analysis_rows)

print("\n=== Abstention Outcome Analysis ===")
display(abstention_analysis_df)


### Printing Always-Answer Baseline Results

This cell filters and prints only the **always-answer** rows from the results table.

- First, we select rows where `method == "always_answer"` from the DataFrame.
- Then, we loop through each sample size (25, 100, 350, 500).
- For each row, we print:
  - `n` (sample size)
  - `THRESH=None` (since baseline does not use a threshold)
  - `coverage` (always 1.00 because it never abstains)
  - `abstain` (always 0)
  - `accuracy` (baseline performance)

This allows us to directly compare the baseline accuracy with the abstention results in the same output format.


In [ ]:
always_df = df[df["method"] == "always_answer"]

for _, row in always_df.iterrows():
    print(
        f"n={int(row['sample_size']):>3} | "
        f"THRESH= None | "
        f"coverage={row['coverage']:.2f} | "
        f"abstain={int(row['abstained']):>3} | "
        f"acc={row['accuracy']}"
    )


In [ ]:
# ===== Cell 1: Imports + Setup =====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

df = df.copy()
df["threshold"] = pd.to_numeric(df["threshold"], errors="coerce")

# Consistent color mapping for sample sizes
COLOR_MAP = {
    25: "#1f77b4",    # blue
    100: "#ff7f0e",   # orange
    350: "#2ca02c",   # green
    500: "#d62728"    # red
}

### Margin Distribution for Correct vs Incorrect Predictions

This cell analyzes whether the log-probability margin is a meaningful confidence signal.

For each question:
- we compute the margin between the top-1 and top-2 answer-letter log-probabilities
- we group margins by whether the model’s prediction was correct or incorrect

If the confidence signal is useful, correct predictions should generally have higher margins,
while incorrect predictions should be concentrated at lower margins.

In [ ]:
# ===== Margin Distribution: Correct vs Incorrect =====

margin_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for i, ex in enumerate(data_mc):
        gold, pred, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)

        margin_rows.append({
            "sample_size": n,
            "example_index": i,
            "margin": margin,
            "is_correct": pred == gold,
            "outcome": "Correct" if pred == gold else "Incorrect"
        })

margin_df = pd.DataFrame(margin_rows)

print("\n=== Margin Summary: Correct vs Incorrect ===")
display(
    margin_df.groupby(["sample_size", "outcome"])["margin"].agg(["count", "mean", "median", "std"])
)

margin_500 = margin_df[margin_df["sample_size"] == 500]

plt.figure(figsize=(8,5))

plt.hist(
    margin_500[margin_500["outcome"] == "Correct"]["margin"],
    bins=30,
    alpha=0.6,
    label="Correct",
    edgecolor="black"
)

plt.hist(
    margin_500[margin_500["outcome"] == "Incorrect"]["margin"],
    bins=30,
    alpha=0.6,
    label="Incorrect",
    edgecolor="black"
)

plt.xlabel("Margin (top-1 log-prob − top-2 log-prob)")
plt.ylabel("Number of Questions")
plt.title("Margin Distribution for Correct vs Incorrect Predictions (n=500)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "margin_correct_vs_incorrect.png"), dpi=200)
plt.show()

In [ ]:
# ===== Dataset Plot 1: Distribution of Number of Choices =====
choice_counts = [
    len(ds_mc["validation"][i]["mc1_targets"]["choices"])
    for i in range(len(ds_mc["validation"]))
]

plt.figure(figsize=(7,5))
plt.hist(
    choice_counts,
    bins=np.arange(min(choice_counts), max(choice_counts)+2)-0.5,
    color="#4C72B0",
    edgecolor="black"
)

plt.xlabel("Number of answer choices (mc1)")
plt.ylabel("Number of questions")
plt.title("TruthfulQA — Distribution of Answer Choices")
plt.xlim(min(choice_counts)-1, max(choice_counts)+1)
plt.ylim(0, max(np.bincount(choice_counts)) + 20)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "dataset_choice_distribution.png"), dpi=200)
plt.show()

print("Explanation: This histogram shows how many answer options each question has.")
print("It indicates the variability in multiple-choice difficulty and structure across the dataset.")

In [ ]:
# ===== Dataset Plot 2: Question Length Distribution =====
q_token_lens = [
    len(tokenizer.encode(ds_mc["validation"][i]["question"], add_special_tokens=False))
    for i in range(len(ds_mc["validation"]))
]

plt.figure(figsize=(7,5))
plt.hist(q_token_lens, bins=30, color="#55A868", edgecolor="black")

plt.xlabel("Question length (tokens)")
plt.ylabel("Number of questions")
plt.title("TruthfulQA — Question Length Distribution")
plt.xlim(0, max(q_token_lens) + 10)
plt.ylim(0, None)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "dataset_question_length.png"), dpi=200)
plt.show()

print("Explanation: This plot shows the distribution of question lengths in tokens.")
print("It helps characterize dataset complexity and the reasoning depth required by the model.")

In [ ]:
# ===== Results Plot 1: Baseline Accuracy vs Sample Size =====
always = df[df["method"] == "always_answer"].sort_values("sample_size")

plt.figure(figsize=(7,5))
plt.plot(
    always["sample_size"],
    always["accuracy"],
    marker="o",
    linewidth=2,
    color="#4C72B0"
)

plt.xlabel("Sample size (n)")
plt.ylabel("Accuracy")
plt.title("Always-Answer Baseline Accuracy vs Sample Size")
plt.xlim(min(always["sample_size"]) - 20, max(always["sample_size"]) + 20)
plt.ylim(0.7, 1.0)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "baseline_accuracy_vs_n.png"), dpi=200)
plt.show()

print("Explanation: This plot shows baseline accuracy when the model is forced to answer every question.")
print("It demonstrates performance stability as evaluation size increases.")

In [ ]:
abst = df[df["method"] == "abstain_margin"].sort_values(["sample_size", "threshold"])

In [ ]:
# ===== Results Plot 2: Accuracy–Coverage Tradeoff (n=500 only) =====
abst_500 = abst[abst["sample_size"] == 500]

plt.figure(figsize=(7,5))

plt.plot(
    abst_500["coverage"],
    abst_500["accuracy"],
    marker="o",
    linewidth=2,
    color=COLOR_MAP[500],
    label="Abstain (n=500)"
)

# Always-answer baseline
baseline_acc = always[always["sample_size"] == 500]["accuracy"].values[0]

plt.axhline(
    y=baseline_acc,
    linestyle="--",
    color="black",
    linewidth=2,
    label="Always-answer baseline"
)

plt.xlabel("Coverage (fraction answered)")
plt.ylabel("Accuracy (on answered)")
plt.title("Accuracy–Coverage Tradeoff (n=500)")
plt.xlim(0.75, 1.02)
plt.ylim(0.75, 1.0)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "accuracy_coverage_tradeoff.png"), dpi=200)
plt.show()

In [ ]:
# ===== Results Plot 3: Accuracy vs Threshold (n=500 only) =====
abst_500 = abst[abst["sample_size"] == 500]

plt.figure(figsize=(7,5))

plt.plot(
    abst_500["threshold"],
    abst_500["accuracy"],
    marker="o",
    linewidth=2,
    color=COLOR_MAP[500],
    label="Abstain (n=500)"
)

# Always-answer baseline
baseline_acc = always[always["sample_size"] == 500]["accuracy"].values[0]

plt.axhline(
    y=baseline_acc,
    linestyle="--",
    color="black",
    linewidth=2,
    label="Always-answer baseline"
)

plt.xlabel("Threshold τ (log-prob margin)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Confidence Threshold (n=500)")
plt.xlim(0, max(abst_500["threshold"]) + 1)
plt.ylim(0.75, 1.0)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "accuracy_vs_threshold.png"), dpi=200)
plt.show()

In [ ]:
# ===== Results Plot 4: Coverage vs Threshold (n=500 only) =====
abst_500 = abst[abst["sample_size"] == 500]

plt.figure(figsize=(7,5))

plt.plot(
    abst_500["threshold"],
    abst_500["coverage"],
    marker="o",
    linewidth=2,
    color=COLOR_MAP[500],
    label="Abstain (n=500)"
)

# Always-answer baseline
plt.axhline(
    y=1.0,
    linestyle="--",
    color="black",
    linewidth=2,
    label="Always-answer baseline"
)

plt.xlabel("Threshold τ (log-prob margin)")
plt.ylabel("Coverage")
plt.title("Coverage vs Confidence Threshold (n=500)")
plt.xlim(0, max(abst_500["threshold"]) + 1)
plt.ylim(0.75, 1.02)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "coverage_vs_threshold.png"), dpi=200)
plt.show()

In [ ]:
# ===== Results Plot 5: Abstentions vs Threshold (n=500 only) =====
abst_500 = abst[abst["sample_size"] == 500]

plt.figure(figsize=(7,5))

plt.plot(
    abst_500["threshold"],
    abst_500["abstained"],
    marker="o",
    linewidth=2,
    color=COLOR_MAP[500],
    label="Abstain (n=500)"
)

# Always-answer baseline (0 abstentions)
plt.axhline(
    y=0,
    linestyle="--",
    color="black",
    linewidth=2,
    label="Always-answer baseline"
)

plt.xlabel("Threshold τ (log-prob margin)")
plt.ylabel("Number of Abstentions")
plt.title("Abstentions vs Confidence Threshold (n=500)")
plt.xlim(0, max(abst_500["threshold"]) + 1)
plt.ylim(0, max(abst_500["abstained"]) + 10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstentions_vs_threshold.png"), dpi=200)
plt.show()

### Visualization of Abstention Outcomes

This plot shows what happens on questions where the model abstains.

For each threshold:
- Blue line: number of abstained questions that would have been **correct**
- Red line: number of abstained questions that would have been **incorrect**

If the abstention strategy is effective, the number of incorrect answers should dominate,
indicating that the model is selectively abstaining on risky predictions rather than
discarding many correct answers.

In [ ]:
# ===== Results Plot 6: Abstention Outcome Breakdown =====

analysis_500 = abstention_analysis_df[
    abstention_analysis_df["sample_size"] == 500
]

plt.figure(figsize=(7,5))

plt.plot(
    analysis_500["threshold"],
    analysis_500["abstained_would_be_correct"],
    marker="o",
    linewidth=2,
    label="Would-be correct",
)

plt.plot(
    analysis_500["threshold"],
    analysis_500["abstained_would_be_wrong"],
    marker="o",
    linewidth=2,
    label="Would-be wrong",
)

plt.xlabel("Threshold τ (log-prob margin)")
plt.ylabel("Number of Abstained Questions")
plt.title("What Happens on Abstained Questions (n=500)")
plt.xlim(0, max(analysis_500["threshold"]) + 1)
plt.ylim(0, max(analysis_500["abstained_total"]) + 10)

plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstention_outcome_breakdown.png"), dpi=200)
plt.show()

### Additional Evaluation Metrics: Precision, Recall, and F1 Score

This cell computes additional classification metrics for the answering behavior of the model.

For the subset of questions the model chooses to answer:
- **Precision** measures how often the model’s answered predictions are correct.
- **Recall** measures how many correct answers from the dataset are successfully retrieved by the model.
- **F1 score** combines precision and recall into a single metric.

Because our system can abstain, recall reflects how many correct answers remain after abstention filtering.
Higher thresholds may increase precision but reduce recall due to more abstentions.

In [ ]:
# ===== Additional Evaluation Metrics: Precision, Recall, F1 =====

metric_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:
        TP = 0   # correct answered
        FP = 0   # wrong answered
        FN = 0   # abstained but correct answer existed

        for ex in data_mc:
            gold, pred, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)

            if margin < t:
                # abstention counts as missed opportunity for recall
                FN += 1
                continue

            if pred == gold:
                TP += 1
            else:
                FP += 1

        precision = TP / (TP + FP) if (TP + FP) > 0 else None
        recall = TP / (TP + FN) if (TP + FN) > 0 else None

        if precision is not None and recall is not None and (precision + recall) > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = None

        metric_rows.append({
            "sample_size": n,
            "threshold": t,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
        })

        print(
            f"n={n:>3} | THRESH={t:>4} | "
            f"precision={precision:.3f} | "
            f"recall={recall:.3f} | "
            f"f1={f1:.3f}"
        )

metrics_df = pd.DataFrame(metric_rows)

print("\n=== Precision / Recall / F1 Summary ===")
display(metrics_df)

metrics_df.to_csv("precision_recall_f1_results.csv", index=False)
print("\nSaved: precision_recall_f1_results.csv")

### Baseline Error Overlap with Abstentions

This cell checks whether questions that were answered incorrectly by the always-answer baseline
are later abstained on by the hallucination detector.

For each threshold:
- we first identify the set of questions the baseline got wrong
- then we check how many of those same questions are abstained by the margin-based detector

This helps measure whether the abstention mechanism is successfully catching baseline mistakes.
If the overlap is high, the detector is effectively abstaining on questions that would otherwise
produce incorrect answers.

In [ ]:
# ===== Baseline Wrong Answers vs Abstained Questions =====

overlap_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    baseline_wrong_indices = set()
    margins_by_index = {}

    # Compute baseline wrong set and store margins once
    for i, ex in enumerate(data_mc):
        q = ex["question"]
        choices = ex["mc1_targets"]["choices"]
        labels = ex["mc1_targets"]["labels"]
        gold_index = labels.index(1)
        gold_letter = chr(ord("A") + gold_index)

        baseline_pred = predict_letter_always_answer(q, choices, style=PROMPT_STYLE)
        if baseline_pred is None or baseline_pred != gold_letter:
            baseline_wrong_indices.add(i)

        _, _, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)
        margins_by_index[i] = margin

    total_baseline_wrong = len(baseline_wrong_indices)

    for t in THRESHOLDS:
        abstained_indices = {
            i for i, margin in margins_by_index.items() if margin < t
        }

        overlap = baseline_wrong_indices.intersection(abstained_indices)
        overlap_count = len(overlap)

        overlap_rate = (
            overlap_count / total_baseline_wrong if total_baseline_wrong > 0 else None
        )

        overlap_rows.append({
            "sample_size": n,
            "threshold": t,
            "baseline_wrong_total": total_baseline_wrong,
            "abstained_from_baseline_wrong": overlap_count,
            "overlap_rate": overlap_rate,
        })

        if overlap_rate is not None:
            print(
                f"n={n:>3} | THRESH={t:>4} | "
                f"baseline_wrong={total_baseline_wrong:>3} | "
                f"abstained_from_wrong={overlap_count:>3} | "
                f"overlap_rate={overlap_rate:.3f}"
            )
        else:
            print(
                f"n={n:>3} | THRESH={t:>4} | "
                f"baseline_wrong=  0 | abstained_from_wrong=  0 | overlap_rate=N/A"
            )

baseline_abstention_overlap_df = pd.DataFrame(overlap_rows)

print("\n=== Baseline Wrong / Abstention Overlap ===")
display(baseline_abstention_overlap_df)

baseline_abstention_overlap_df.to_csv("baseline_abstention_overlap.csv", index=False)
print("\nSaved: baseline_abstention_overlap.csv")

In [ ]:
# ===== Improved Results Plot 7: Baseline Errors Caught by Abstention =====

overlap_500 = baseline_abstention_overlap_df[
    baseline_abstention_overlap_df["sample_size"] == 500
].copy()

abst_500 = df[
    (df["method"] == "abstain_margin") &
    (df["sample_size"] == 500)
].sort_values("threshold")

plot_df = overlap_500.merge(
    abst_500[["threshold", "abstained"]],
    on="threshold",
    how="left"
).sort_values("threshold")

fig, ax1 = plt.subplots(figsize=(8,5))

# Left axis: overlap rate
line1 = ax1.plot(
    plot_df["threshold"],
    plot_df["overlap_rate"],
    marker="o",
    linewidth=2,
    label="Fraction of baseline errors caught"
)

ax1.set_xlabel("Threshold τ (log-prob margin)")
ax1.set_ylabel("Fraction of Baseline Errors Abstained")
ax1.set_xlim(0, max(plot_df["threshold"]) + 1)
ax1.set_ylim(0, 1.02)
ax1.grid(True, linestyle="--", alpha=0.6)

# Annotate rate values
for x, y in zip(plot_df["threshold"], plot_df["overlap_rate"]):
    ax1.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9)

# Right axis: counts
ax2 = ax1.twinx()

line2 = ax2.plot(
    plot_df["threshold"],
    plot_df["abstained_from_baseline_wrong"],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="Baseline wrong questions caught"
)

line3 = ax2.plot(
    plot_df["threshold"],
    plot_df["abstained"],
    marker="^",
    linewidth=2,
    linestyle=":",
    label="Total abstentions"
)

ax2.set_ylabel("Number of Questions")
ax2.set_ylim(0, max(plot_df["abstained"]) + 10)

# Combined legend
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="lower right")

plt.title("How Many Baseline Errors Are Caught by Abstention? (n=500)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "baseline_error_overlap_improved.png"), dpi=200)
plt.show()

### Relationship Between Number of Answer Choices and Abstention

This cell analyzes whether the model is more likely to abstain on questions with
a larger number of answer choices.

For each threshold:
- we group questions by how many answer options they have
- we compute the fraction of those questions that were abstained

If abstention increases with the number of choices, it suggests that the model's
confidence decreases as the decision space becomes larger.

In [ ]:
# ===== Abstention vs Number of Answer Choices =====

choice_analysis_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:
        stats = {}

        for ex in data_mc:
            num_choices = len(ex["mc1_targets"]["choices"])

            if num_choices not in stats:
                stats[num_choices] = {"total": 0, "abstained": 0}

            stats[num_choices]["total"] += 1

            gold, pred, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)

            if margin < t:
                stats[num_choices]["abstained"] += 1

        for c in stats:
            total = stats[c]["total"]
            abstained = stats[c]["abstained"]
            rate = abstained / total if total > 0 else None

            choice_analysis_rows.append({
                "sample_size": n,
                "threshold": t,
                "num_choices": c,
                "total_questions": total,
                "abstained": abstained,
                "abstention_rate": rate
            })

choice_abstention_df = pd.DataFrame(choice_analysis_rows)

print("\n=== Abstention vs Number of Choices ===")
display(choice_abstention_df)

choice_abstention_df.to_csv("abstention_vs_choices.csv", index=False)
print("\nSaved: abstention_vs_choices.csv")

In [ ]:
# ===== Results Plot: Abstention Rate vs Number of Answer Choices =====

choice_500 = choice_abstention_df[
    choice_abstention_df["sample_size"] == 500
]

plt.figure(figsize=(7,5))

for t in THRESHOLDS:
    sub = choice_500[choice_500["threshold"] == t]

    plt.plot(
        sub["num_choices"],
        sub["abstention_rate"],
        marker="o",
        linewidth=2,
        label=f"τ={t}"
    )

plt.xlabel("Number of Answer Choices")
plt.ylabel("Abstention Rate")
plt.title("Does More Answer Choices Increase Abstention? (n=500)")
plt.xlim(min(choice_500["num_choices"]) - 0.5, max(choice_500["num_choices"]) + 0.5)
plt.ylim(0, 1.02)

plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstention_vs_num_choices.png"), dpi=200)
plt.show()

In [ ]:
# ===== Improved Plot: Abstention Rate vs Number of Answer Choices (Heatmap) =====

choice_500 = choice_abstention_df[
    choice_abstention_df["sample_size"] == 500
].copy()

heatmap_df = choice_500.pivot(
    index="threshold",
    columns="num_choices",
    values="abstention_rate"
).sort_index()

plt.figure(figsize=(8, 5))
im = plt.imshow(
    heatmap_df,
    aspect="auto",
    interpolation="nearest",
    origin="lower"
)

plt.colorbar(im, label="Abstention Rate")
plt.xticks(
    ticks=np.arange(len(heatmap_df.columns)),
    labels=heatmap_df.columns
)
plt.yticks(
    ticks=np.arange(len(heatmap_df.index)),
    labels=heatmap_df.index
)

plt.xlabel("Number of Answer Choices")
plt.ylabel("Threshold τ")
plt.title("Abstention Rate by Threshold and Number of Choices (n=500)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstention_vs_num_choices_heatmap.png"), dpi=200)
plt.show()

In [ ]:
# ===== Improved Line Plot: Abstention Rate vs Number of Answer Choices =====

choice_500 = choice_abstention_df[
    choice_abstention_df["sample_size"] == 500
]

selected_thresholds = [0.5, 2.0, 5.0, 10.0]

plt.figure(figsize=(8,5))

for t in selected_thresholds:
    sub = choice_500[choice_500["threshold"] == t].sort_values("num_choices")

    plt.plot(
        sub["num_choices"],
        sub["abstention_rate"],
        marker="o",
        linewidth=2,
        label=f"τ={t}"
    )

plt.xlabel("Number of Answer Choices")
plt.ylabel("Abstention Rate")
plt.title("Does More Answer Choices Increase Abstention? (n=500)")
plt.xlim(min(choice_500["num_choices"]) - 0.5, max(choice_500["num_choices"]) + 0.5)
plt.ylim(0, 0.5)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(title="Threshold")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstention_vs_num_choices_clean.png"), dpi=200)
plt.show()

### False Positives and False Negatives by Number of Answer Choices

This cell checks whether detector errors are related to the number of answer options.

For each threshold and each choice count:
- false positive rate measures how often the detector abstains on questions the baseline would have answered correctly
- false negative rate measures how often the detector answers despite the baseline being wrong

This helps determine whether larger decision spaces contribute to under-confidence
(false positives) or over-confidence (false negatives).

In [ ]:
# ===== FP / FN by Number of Answer Choices =====

choice_case_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:
        stats = {}

        for ex in data_mc:
            num_choices = len(ex["mc1_targets"]["choices"])

            if num_choices not in stats:
                stats[num_choices] = {"TP": 0, "FP": 0, "FN": 0, "TN": 0}

            q = ex["question"]
            choices = ex["mc1_targets"]["choices"]
            labels = ex["mc1_targets"]["labels"]

            gold_index = labels.index(1)
            gold_letter = chr(ord("A") + gold_index)

            baseline_pred = predict_letter_always_answer(q, choices, style=PROMPT_STYLE)
            baseline_wrong = (baseline_pred != gold_letter)

            _, _, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)
            abstain = margin < t

            if abstain and baseline_wrong:
                stats[num_choices]["TP"] += 1
            elif abstain and not baseline_wrong:
                stats[num_choices]["FP"] += 1
            elif not abstain and baseline_wrong:
                stats[num_choices]["FN"] += 1
            else:
                stats[num_choices]["TN"] += 1

        for c in sorted(stats.keys()):
            TP = stats[c]["TP"]
            FP = stats[c]["FP"]
            FN = stats[c]["FN"]
            TN = stats[c]["TN"]

            fpr = FP / (FP + TN) if (FP + TN) > 0 else None
            fnr = FN / (FN + TP) if (FN + TP) > 0 else None

            choice_case_rows.append({
                "sample_size": n,
                "threshold": t,
                "num_choices": c,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "false_positive_rate": fpr,
                "false_negative_rate": fnr
            })

choice_case_df = pd.DataFrame(choice_case_rows)

print("\n=== FP / FN by Number of Answer Choices ===")
display(choice_case_df.head())

choice_case_500 = choice_case_df[
    (choice_case_df["sample_size"] == 500) &
    (choice_case_df["threshold"] == 5.0)
].sort_values("num_choices")

plt.figure(figsize=(8,5))

plt.plot(
    choice_case_500["num_choices"],
    choice_case_500["false_positive_rate"],
    marker="o",
    linewidth=2,
    label="False Positive Rate"
)

plt.plot(
    choice_case_500["num_choices"],
    choice_case_500["false_negative_rate"],
    marker="s",
    linewidth=2,
    label="False Negative Rate"
)

plt.xlabel("Number of Answer Choices")
plt.ylabel("Rate")
plt.title("FP/FN Rates by Number of Answer Choices (n=500, τ=5.0)")
plt.ylim(0, 1.02)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fp_fn_by_num_choices.png"), dpi=200)
plt.show()

### Question Length and Detector Behavior

This cell analyzes whether longer questions are associated with lower confidence
or worse detector behavior.

We bin questions by token length and compute:
- average margin
- abstention rate
- false positive rate
- false negative rate

This helps test whether question length contributes to under-confidence or
over-confidence in the abstention mechanism.

In [ ]:
# ===== Question Length vs Margin / Abstention / FP / FN =====

length_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:
        for ex in data_mc:
            q = ex["question"]
            choices = ex["mc1_targets"]["choices"]
            labels = ex["mc1_targets"]["labels"]

            q_len = len(tokenizer.encode(q, add_special_tokens=False))

            gold_index = labels.index(1)
            gold_letter = chr(ord("A") + gold_index)

            baseline_pred = predict_letter_always_answer(q, choices, style=PROMPT_STYLE)
            baseline_wrong = (baseline_pred != gold_letter)

            _, _, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)
            abstain = margin < t

            length_rows.append({
                "sample_size": n,
                "threshold": t,
                "question_length": q_len,
                "margin": margin,
                "baseline_wrong": baseline_wrong,
                "abstain": abstain
            })

length_df = pd.DataFrame(length_rows)

# length bins
length_df["length_bin"] = pd.cut(
    length_df["question_length"],
    bins=[0, 10, 20, 30, 1000],
    labels=["0-10", "11-20", "21-30", "31+"],
    include_lowest=True
)

summary_rows = []

for (n, t, b), sub in length_df.groupby(["sample_size", "threshold", "length_bin"], observed=False):
    TP = ((sub["abstain"] == True) & (sub["baseline_wrong"] == True)).sum()
    FP = ((sub["abstain"] == True) & (sub["baseline_wrong"] == False)).sum()
    FN = ((sub["abstain"] == False) & (sub["baseline_wrong"] == True)).sum()
    TN = ((sub["abstain"] == False) & (sub["baseline_wrong"] == False)).sum()

    abstention_rate = sub["abstain"].mean() if len(sub) > 0 else None
    avg_margin = sub["margin"].mean() if len(sub) > 0 else None
    fpr = FP / (FP + TN) if (FP + TN) > 0 else None
    fnr = FN / (FN + TP) if (FN + TP) > 0 else None

    summary_rows.append({
        "sample_size": n,
        "threshold": t,
        "length_bin": str(b),
        "count": len(sub),
        "avg_margin": avg_margin,
        "abstention_rate": abstention_rate,
        "false_positive_rate": fpr,
        "false_negative_rate": fnr
    })

length_summary_df = pd.DataFrame(summary_rows)

print("\n=== Question Length Analysis ===")
display(length_summary_df.head(20))

length_500 = length_summary_df[
    (length_summary_df["sample_size"] == 500) &
    (length_summary_df["threshold"] == 5.0)
].copy()

length_500["length_bin"] = pd.Categorical(
    length_500["length_bin"],
    categories=["0-10", "11-20", "21-30", "31+"],
    ordered=True
)
length_500 = length_500.sort_values("length_bin")

plt.figure(figsize=(8,5))

plt.plot(
    length_500["length_bin"],
    length_500["abstention_rate"],
    marker="o",
    linewidth=2,
    label="Abstention Rate"
)

plt.plot(
    length_500["length_bin"],
    length_500["false_positive_rate"],
    marker="s",
    linewidth=2,
    label="False Positive Rate"
)

plt.plot(
    length_500["length_bin"],
    length_500["false_negative_rate"],
    marker="^",
    linewidth=2,
    label="False Negative Rate"
)

plt.xlabel("Question Length Bin (tokens)")
plt.ylabel("Rate")
plt.title("Detector Behavior by Question Length (n=500, τ=5.0)")
plt.ylim(0, 1.02)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "detector_behavior_by_question_length.png"), dpi=200)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(
    length_500["length_bin"],
    length_500["avg_margin"],
    marker="o",
    linewidth=2
)

plt.xlabel("Question Length Bin (tokens)")
plt.ylabel("Average Margin")
plt.title("Average Margin by Question Length (n=500, τ=5.0)")
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "avg_margin_by_question_length.png"), dpi=200)
plt.show()

### Abstention vs FP/FN Tradeoff


False Positives (FP)
False positives occur when the system abstains on questions that the model would have answered correctly. These represent unnecessary refusals and reduce coverage.

False Negatives (FN)
False negatives occur when the system answers a question despite low confidence and produces an incorrect answer. These correspond to hallucinations that were not filtered by the abstention mechanism.

In [ ]:
# ===== Compute FP / FN for Abstention Detector =====

confusion_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:
        TP = FP = FN = TN = 0

        for ex in data_mc:

            q = ex["question"]
            choices = ex["mc1_targets"]["choices"]
            labels = ex["mc1_targets"]["labels"]
            gold_index = labels.index(1)
            gold_letter = chr(ord("A") + gold_index)

            baseline_pred = predict_letter_always_answer(q, choices, style=PROMPT_STYLE)
            baseline_wrong = (baseline_pred != gold_letter)

            _, _, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)
            abstain = margin < t

            if abstain and baseline_wrong:
                TP += 1
            elif abstain and not baseline_wrong:
                FP += 1
            elif not abstain and baseline_wrong:
                FN += 1
            else:
                TN += 1

        abstention_rate = (TP + FP) / (TP + FP + FN + TN)
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        fnr = FN / (FN + TP) if (FN + TP) > 0 else 0

        confusion_rows.append({
            "sample_size": n,
            "threshold": t,
            "abstention_rate": abstention_rate,
            "false_positive_rate": fpr,
            "false_negative_rate": fnr
        })

confusion_df = pd.DataFrame(confusion_rows)
display(confusion_df)

In [ ]:
# ===== Plot: Abstention Rate with FP and FN =====

conf_500 = confusion_df[confusion_df["sample_size"] == 500]

plt.figure(figsize=(8,5))

plt.plot(
    conf_500["threshold"],
    conf_500["abstention_rate"],
    marker="o",
    linewidth=2,
    label="Abstention Rate"
)

plt.plot(
    conf_500["threshold"],
    conf_500["false_positive_rate"],
    marker="s",
    linewidth=2,
    label="False Positive Rate"
)

plt.plot(
    conf_500["threshold"],
    conf_500["false_negative_rate"],
    marker="^",
    linewidth=2,
    label="False Negative Rate"
)

plt.xlabel("Threshold τ (log-prob margin)")
plt.ylabel("Rate")
plt.title("Abstention vs False Positives and False Negatives (n=500)")

plt.ylim(0,1)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstention_fp_fn_tradeoff.png"), dpi=200)
plt.show()

### Margin Distribution for TP, FP, FN, and TN Cases

This cell studies how the confidence margin behaves across the four detector outcomes:

- TP: abstain and baseline answer would be wrong
- FP: abstain and baseline answer would be correct
- FN: answer and baseline answer is wrong
- TN: answer and baseline answer is correct

This helps explain detector failures:
- false positives correspond to under-confident correct answers
- false negatives correspond to over-confident incorrect answers

In [ ]:
# ===== Margin Distribution by TP / FP / FN / TN =====

CASE_THRESHOLD = 5.0   # fixed threshold for case analysis; can change to 6.0 or 8.0 if desired

case_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for i, ex in enumerate(data_mc):
        q = ex["question"]
        choices = ex["mc1_targets"]["choices"]
        labels = ex["mc1_targets"]["labels"]

        gold_index = labels.index(1)
        gold_letter = chr(ord("A") + gold_index)

        baseline_pred = predict_letter_always_answer(q, choices, style=PROMPT_STYLE)
        baseline_wrong = (baseline_pred != gold_letter)

        _, _, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)
        abstain = margin < CASE_THRESHOLD

        if abstain and baseline_wrong:
            case = "TP"
        elif abstain and not baseline_wrong:
            case = "FP"
        elif not abstain and baseline_wrong:
            case = "FN"
        else:
            case = "TN"

        case_rows.append({
            "sample_size": n,
            "example_index": i,
            "threshold": CASE_THRESHOLD,
            "margin": margin,
            "case": case
        })

case_df = pd.DataFrame(case_rows)

print(f"\n=== Case Summary at threshold {CASE_THRESHOLD} ===")
display(case_df.groupby(["sample_size", "case"])["margin"].agg(["count", "mean", "median", "std"]))

case_500 = case_df[case_df["sample_size"] == 500]

plt.figure(figsize=(8,5))

for case_name in ["TP", "FP", "FN", "TN"]:
    sub = case_500[case_500["case"] == case_name]
    if len(sub) > 0:
        plt.hist(
            sub["margin"],
            bins=30,
            alpha=0.5,
            label=case_name,
            edgecolor="black"
        )

plt.xlabel("Margin (top-1 log-prob − top-2 log-prob)")
plt.ylabel("Number of Questions")
plt.title(f"Margin Distribution by Detector Outcome (n=500, τ={CASE_THRESHOLD})")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "margin_tp_fp_fn_tn.png"), dpi=200)
plt.show()

### ROC Curve

ROC-style Analysis

This figure evaluates the margin-based abstention mechanism as a binary detector for incorrect predictions. The true positive rate measures how often the system abstains on questions the baseline would answer incorrectly, while the false positive rate measures how often it abstains on questions the baseline would answer correctly. Points further toward the top-left indicate better separation between correct and incorrect predictions.

In [ ]:
# ===== Compute ROC-style statistics for abstention detector =====

roc_rows = []

for n in SAMPLE_SIZES:
    data_mc = ds_mc["validation"].select(range(min(n, len(ds_mc["validation"]))))

    for t in THRESHOLDS:

        TP = FP = FN = TN = 0

        for ex in data_mc:

            q = ex["question"]
            choices = ex["mc1_targets"]["choices"]
            labels = ex["mc1_targets"]["labels"]
            gold_index = labels.index(1)
            gold_letter = chr(ord("A") + gold_index)

            # baseline prediction
            baseline_pred = predict_letter_always_answer(q, choices, style=PROMPT_STYLE)
            baseline_wrong = (baseline_pred != gold_letter)

            # abstention decision
            _, _, margin = predict_with_confidence_aligned(ex, style=PROMPT_STYLE)
            abstain = margin < t

            if abstain and baseline_wrong:
                TP += 1
            elif abstain and not baseline_wrong:
                FP += 1
            elif not abstain and baseline_wrong:
                FN += 1
            else:
                TN += 1

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        roc_rows.append({
            "sample_size": n,
            "threshold": t,
            "TPR": TPR,
            "FPR": FPR
        })

roc_df = pd.DataFrame(roc_rows)

print("\n=== ROC Data ===")
display(roc_df)

In [ ]:
# ===== ROC-style Curve for Abstention Detector =====

roc_500 = roc_df[roc_df["sample_size"] == 500].sort_values("FPR")

plt.figure(figsize=(7,6))

plt.plot(
    roc_500["FPR"],
    roc_500["TPR"],
    marker="o",
    linewidth=2,
    label="Abstention Detector"
)

# random baseline
plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    color="gray",
    label="Random"
)

# annotate thresholds
for _, row in roc_500.iterrows():
    plt.annotate(
        f"τ={row['threshold']}",
        (row["FPR"], row["TPR"]),
        textcoords="offset points",
        xytext=(5,5),
        fontsize=9
    )

plt.xlabel("False Positive Rate (Abstain on Correct Answers)")
plt.ylabel("True Positive Rate (Abstain on Wrong Answers)")
plt.title("ROC-style Curve for Confidence-based Abstention (n=500)")

plt.xlim(0,1)
plt.ylim(0,1)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "abstention_roc_curve.png"), dpi=200)
plt.show()

from sklearn.metrics import auc
auc_score = auc(roc_500["FPR"], roc_500["TPR"])
print("ROC AUC:", auc_score)